## Load Clean Dataset

The cleaned complaint dataset generated in Task 1 is loaded as the input for Task 2.

This dataset contains:
- Complaint information
- Product category
- Complaint narrative
- Cleaned text used for embedding generation

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/filtered_complaints.csv"
)

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,narrative_length,clean_text
0,2025-06-13,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,A XXXX XXXX card was opened under my name by a...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78230,Servicemember,Consent provided,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN,14069121,91,a xxxx xxxx card was opened under my name by a...
1,2025-06-13,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,I made the mistake of using my wellsfargo debi...,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,ID,83815,NaN,Consent provided,Web,2025-06-13,Closed with explanation,Yes,NaN,14061897,109,i made the mistake of using my wellsfargo debi...
2,2025-06-12,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,"Dear CFPB, I have a secured credit card with c...",Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11220,NaN,Consent provided,Web,2025-06-13,Closed with monetary relief,Yes,NaN,14047085,156,dear cfpb i have a secured credit card with ci...
3,2025-06-12,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,I have a Citi rewards cards. The credit balanc...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",IL,60067,NaN,Consent provided,Web,2025-06-12,Closed with explanation,Yes,NaN,14040217,233,i have a citi rewards cards the credit balance...
4,2025-06-09,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,b'I am writing to dispute the following charge...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78413,Older American,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13968411,454,b i am writing to dispute the following charge...


## Product Category Distribution

The cleaned dataset contains four financial product categories selected during Task 1. Complaint counts vary across categories, with **Checking or Savings Account** having the largest number of complaints and **Payday Loan, Title Loan, Personal Loan, or Advance Loan** having the fewest. This distribution will be preserved during stratified sampling in Task 2 to ensure each product category is proportionally represented in the sample.

In [2]:
df["Product"].value_counts()

Product
Checking or savings account                                140319
Money transfer, virtual currency, or money service          97188
Credit card                                                 80667
Payday loan, title loan, personal loan, or advance loan      8896
Name: count, dtype: int64

#Sampling Strategy

In [20]:
sample_size = 12000

sample_df = pd.concat(
    [
        group.sample(
            n=int(sample_size * len(group) / len(df)),
            random_state=42
        )
        for name, group in df.groupby("Product")
    ]
).reset_index(drop=True)

## Sampling Strategy

The original processed dataset contains 80,667 complaint records. Due to local hardware memory limitations, a smaller subset was used for embedding generation and FAISS indexing.

For efficient local processing, 10,000 records were loaded from the processed dataset, and a sample of 1,000 complaint records was selected for building the retrieval system.

The sampled dataset was used for:
- Text chunking
- Sentence Transformer embedding generation
- FAISS vector indexing

This approach reduces memory requirements while maintaining a functional RAG retrieval pipeline. The full processed dataset was retained, and the pipeline can be scaled to larger datasets with additional computational resources.

In [21]:
print(df.columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'narrative_length', 'clean_text']


In [22]:
print(sample_df.columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'narrative_length', 'clean_text']


In [23]:
sample_df["Product"].value_counts()

Product
Checking or savings account                                5148
Money transfer, virtual currency, or money service         3565
Credit card                                                2959
Payday loan, title loan, personal loan, or advance loan     326
Name: count, dtype: int64

## Save Sample Dataset

In [24]:
sample_df.to_csv(
    "../data/sample_complaints.csv",
    index=False
)

In [25]:
sample_df.shape

(11998, 20)

#Text Chunking

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\admin\Desktop\10-academy\rag-complaint-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [28]:
chunks = []

for _, row in sample_df.iterrows():

    texts = splitter.split_text(
        row["clean_text"]
    )

    for text in texts:
        chunks.append(
            {
                "complaint_id": row["Complaint ID"],
                "product": row["Product"],
                "text": text
            }
        )

In [29]:
chunks_df = pd.DataFrame(chunks)

chunks_df.head()

,complaint_id,product,text
0,11750738,Checking or savings account,bank located in has received my check of showe...
1,7166095,Checking or savings account,overdraft fees being charged and atm fees bein...
2,8225373,Checking or savings account,i am writing to formally submit a complaint ag...
3,8225373,Checking or savings account,dispute claim xxxx capital one s response was ...
4,8225373,Checking or savings account,the situation escalated when i discovered anot...


In [30]:
chunks_df.shape

(33801, 3)

## Chunking Strategy

Complaint narratives were split into smaller text chunks using LangChain's RecursiveCharacterTextSplitter.

Configuration:

- Chunk size: 500 characters
- Chunk overlap: 50 characters

A chunk size of 500 characters was selected because complaint narratives usually contain short descriptions of customer issues. This size provides enough context for semantic retrieval while avoiding overly large text segments that may reduce retrieval accuracy.

The 50-character overlap preserves important information between neighboring chunks, reducing the chance of losing context at chunk boundaries.

## Generate Embeddings

In [31]:
from sentence_transformers import SentenceTransformer


model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2062.20it/s]


## Embedding Model Selection

The `sentence-transformers/all-MiniLM-L6-v2` model was selected for generating text embeddings.

This model provides a good balance between semantic search quality and computational efficiency. It produces 384-dimensional embeddings and is lightweight enough to run locally while maintaining strong performance for similarity search tasks.

In [32]:
embeddings = model.encode(
    chunks_df["text"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 1057/1057 [39:00<00:00,  2.21s/it]


In [33]:
embeddings.shape

(33801, 384)

## Create FAISS Vector Store

In [34]:
import faiss
import numpy as np

In [35]:
embedding_array = np.array(
    embeddings
).astype("float32")

In [36]:
dimension = embedding_array.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    embedding_array
)

In [37]:
index.ntotal

33801

## Save Vector Store

In [38]:
faiss.write_index(
    index,
    "../vector_store/faiss.index"
)

In [39]:
chunks_df.to_csv(
    "../vector_store/metadata.csv",
    index=False
)